In [1]:
# Stage 5 — PPE Representation Study
# 5.1 — Spatiotemporal Embedding (A′)
#
# Goal: Build a per-plant delta matrix (n_common_bins × 52) from the PPE
# opportunity surface, fit PCA to 4D, and produce V_delta embeddings.
# Final feature vector: [Vf (15D), Vp (15D), N (1D), V_delta (4D)] = 35D
# Logistic regression trained and compared against A2 (31D) and A3 (32D).

import numpy as np
import pandas as pd
import glob
import pickle
from sklearn.decomposition import PCA

BASE = "/scratch/ariana.l/Stage 4 Link Prediction Model/"
PPE_DIR = "/scratch/ariana.l/ppe-outputs/opportunity_surface/"

# --- Reconstruct common bins ---
print("Loading existence matrices to get common bins...")
F = pd.read_csv(BASE + "stage4_F_existence_phenofield.csv", index_col=0)
P = pd.read_csv(BASE + "stage4_P_existence_gbif_combined.csv", index_col=0)

common_bins = sorted(set(F.columns) & set(P.columns))
print(f"Common bins: {len(common_bins)}")
print(f"Sample bin keys: {common_bins[:5]}")

Loading existence matrices to get common bins...
Common bins: 3160
Sample bin keys: ['24.5_-81.0', '24.5_-81.5', '24.5_-82.0', '24.5_-82.5', '24.5_-83.0']


In [3]:
# Cell 2 — Build delta matrix (n_species × 3160 × 52) from PPE opportunity surface

bin_to_idx = {b: i for i, b in enumerate(common_bins)}
n_bins = len(common_bins)  # 3160
n_weeks = 52

files = sorted(glob.glob(PPE_DIR + "part_*.parquet"))
print(f"Found {len(files)} parquet files")

# Check bin key format from one file first
sample = pd.read_parquet(files[0], columns=['species', 'centroid_lat', 'centroid_lon', 'week', 'norm'])
sample['bin_key'] = sample['centroid_lat'].astype(str) + '_' + sample['centroid_lon'].astype(str)
print("Sample PPE bin keys:", sample['bin_key'].unique()[:5])
print("Sample common_bins:", common_bins[:5])

Found 6697 parquet files
Sample PPE bin keys: ['37.75_-122.25' '38.75_-77.25' '37.25_-122.25' '42.25_-71.25'
 '47.75_-122.25']
Sample common_bins: ['24.5_-81.0', '24.5_-81.5', '24.5_-82.0', '24.5_-82.5', '24.5_-83.0']


In [4]:
# Cell 3 — Build delta matrix from PPE opportunity surface
#
# NOTE: Grid offset between existence matrices (F, P) and PPE opportunity surface.
# F/P bins use 0.5° centroids anchored at .0 and .5 (e.g. 24.5, 25.0).
# PPE centroids are anchored at .25 and .75 (e.g. 37.75, 38.25).
# Both are 0.5° grids offset by 0.25°. We snap PPE centroids to the nearest
# F/P bin via round(x * 2) / 2. Spatial error is at most 0.25°, acceptable
# at this resolution. Flag for Dan.

from collections import defaultdict

def snap(x):
    return round(round(x * 2) / 2, 1)

def fmt(x):
    return f"{x:.1f}"

# Rebuild bin_to_idx using original common_bins format
bin_to_idx = {b: i for i, b in enumerate(common_bins)}

files = sorted(glob.glob(PPE_DIR + "part_*.parquet"))
print(f"Found {len(files)} parquet files")

species_data = defaultdict(list)  # species -> list of (bin_idx, week_idx, norm)

print("Reading opportunity surface files...")
for i, fpath in enumerate(files):
    df = pd.read_parquet(fpath, columns=['species', 'centroid_lat', 'centroid_lon', 'week', 'norm'])
    df['bin_key'] = df['centroid_lat'].map(snap).map(fmt) + '_' + df['centroid_lon'].map(snap).map(fmt)
    df = df[df['bin_key'].isin(bin_to_idx)]
    for row in df.itertuples(index=False):
        species_data[row.species].append((bin_to_idx[row.bin_key], int(row.week) - 1, row.norm))
    if (i + 1) % 500 == 0:
        print(f"  {i+1}/{len(files)} files, {len(species_data)} species seen so far")

print(f"\nDone. Species with PPE data: {len(species_data)}")

Found 6697 parquet files
Reading opportunity surface files...
  500/6697 files, 500 species seen so far
  1000/6697 files, 1000 species seen so far
  1500/6697 files, 1500 species seen so far
  2000/6697 files, 2000 species seen so far
  2500/6697 files, 2500 species seen so far
  3000/6697 files, 3000 species seen so far
  3500/6697 files, 3500 species seen so far
  4000/6697 files, 4000 species seen so far
  4500/6697 files, 4500 species seen so far
  5000/6697 files, 5000 species seen so far
  5500/6697 files, 5500 species seen so far
  6000/6697 files, 6000 species seen so far
  6500/6697 files, 6500 species seen so far

Done. Species with PPE data: 6697


In [5]:
# Cell 4 — Assemble delta matrix and fit PCA to 4D

plant_species = sorted(species_data.keys())
n_species = len(plant_species)
print(f"Plant species with PPE data: {n_species}")

# Allocate (n_species × n_bins × n_weeks), zero-initialized
print(f"Allocating delta matrix: ({n_species}, {n_bins}, {n_weeks})...")
delta_3d = np.zeros((n_species, n_bins, n_weeks), dtype=np.float32)

print("Filling delta matrix...")
for sp_i, sp in enumerate(plant_species):
    for (bin_idx, week_idx, norm) in species_data[sp]:
        delta_3d[sp_i, bin_idx, week_idx] = norm
    if (sp_i + 1) % 1000 == 0:
        print(f"  {sp_i+1}/{n_species} species filled")

print(f"Delta matrix shape: {delta_3d.shape}")
print(f"Memory usage: {delta_3d.nbytes / 1e9:.2f} GB")

# Flatten to (n_species, n_bins * n_weeks) for PCA
print("Flattening...")
delta_flat = delta_3d.reshape(n_species, n_bins * n_weeks)
print(f"Flattened shape: {delta_flat.shape}")

# Fit PCA to 4D
print("Fitting PCA (randomized, 4 components)...")
pca_delta = PCA(n_components=4, svd_solver='randomized', random_state=42)
V_delta = pca_delta.fit_transform(delta_flat)
print(f"V_delta shape: {V_delta.shape}")
print(f"Variance explained: {pca_delta.explained_variance_ratio_}")
print(f"Total variance explained: {pca_delta.explained_variance_ratio_.sum():.3f}")

Plant species with PPE data: 6697
Allocating delta matrix: (6697, 3160, 52)...
Filling delta matrix...
  1000/6697 species filled
  2000/6697 species filled
  3000/6697 species filled
  4000/6697 species filled
  5000/6697 species filled
  6000/6697 species filled
Delta matrix shape: (6697, 3160, 52)
Memory usage: 4.40 GB
Flattening...
Flattened shape: (6697, 164320)
Fitting PCA (randomized, 4 components)...
V_delta shape: (6697, 4)
Variance explained: [0.20070495 0.10929918 0.05240155 0.02480358]
Total variance explained: 0.387


In [6]:
# Cell 5 — Save V_delta and build species index

STAGE5_BASE = "/scratch/ariana.l/Stage 5 PPE Representation Study/"
import os
os.makedirs(STAGE5_BASE, exist_ok=True)

# Save V_delta as DataFrame with species index
Vd_df = pd.DataFrame(V_delta, index=plant_species, columns=[f'PC{i+1}' for i in range(4)])
Vd_df.to_csv(STAGE5_BASE + "stage5_Vdelta_ppe.csv")
print(f"Saved V_delta: {Vd_df.shape}")

# Save PCA object
with open(STAGE5_BASE + "stage5_pca_delta.pkl", "wb") as f:
    pickle.dump(pca_delta, f)
print("Saved PCA object")

# Free the large arrays — no longer needed
del delta_3d, delta_flat
import gc; gc.collect()
print("Freed delta_3d and delta_flat from memory")

print(f"\nV_delta sample:\n{Vd_df.head()}")

Saved V_delta: (6697, 4)
Saved PCA object
Freed delta_3d and delta_flat from memory

V_delta sample:
                            PC1       PC2       PC3       PC4
Abdra brachycarpa    -11.513473  1.062398  9.118951  1.438696
Abronia ameliae       -1.249614 -6.861928 -2.063063 -1.651187
Abronia angustifolia   1.989026  0.252059 -0.901939  0.391061
Abronia elliptica     -5.399944  4.093888 -0.839950 -0.386964
Abronia fragrans      -1.941552  4.970646 -3.387736 -0.590002


In [7]:
# Cell 6 — Reconstruct training pairs from GloBI (same logic as Stage 4)

print("Loading GloBI interactions...")
globi = pd.read_csv(BASE + "stage4_globi_conus_broad.csv")
print(f"GloBI shape: {globi.shape}")
print(f"Columns: {globi.columns.tolist()}")
print(globi.head(3))

Loading GloBI interactions...
GloBI shape: (715215, 9)
Columns: ['sourceTaxonName', 'sourceTaxonFamilyName', 'sourceTaxonOrderName', 'interactionTypeName', 'targetTaxonName', 'targetTaxonFamilyName', 'targetTaxonOrderName', 'decimalLatitude', 'decimalLongitude']
    sourceTaxonName sourceTaxonFamilyName sourceTaxonOrderName  \
0  Bombus sylvicola                Apidae          Hymenoptera   
1  Bombus balteatus                Apidae          Hymenoptera   
2  Bombus sylvicola                Apidae          Hymenoptera   

  interactionTypeName        targetTaxonName targetTaxonFamilyName  \
0          pollinates      Lupinus monticola              Fabaceae   
1          pollinates   Castilleja pulchella         Orobanchaceae   
2          pollinates  Trifolium dasyphyllum              Fabaceae   

  targetTaxonOrderName  decimalLatitude  decimalLongitude  
0              Fabales             45.0       -109.416667  
1             Lamiales             45.0       -109.416667  
2          

In [9]:
# Cell 7 (fixed) — align F and P to common bins before computing N

# Align existence matrices to common bins
F_common = F[common_bins]  # (6466, 3160)
P_common = P[common_bins]  # (4515, 3160)

print(f"F_common: {F_common.shape}, P_common: {P_common.shape}")

def build_features(row):
    vf = Vf_df.loc[row.plant].values          # 15D
    vp = Vp_df.loc[row.pollinator].values     # 15D
    vd = Vd_df.loc[row.plant].values          # 4D
    N = float(np.dot(F_common.loc[row.plant].values, P_common.loc[row.pollinator].values))
    return np.concatenate([vf, vp, [N], vd])

print("Assembling feature matrix...")
X = np.vstack([build_features(row) for row in pairs.itertuples()])
y = pairs['label'].values
print(f"X shape: {X.shape}, y shape: {y.shape}, positive rate: {y.mean():.3f}")

F_common: (6466, 3160), P_common: (4515, 3160)
Assembling feature matrix...
X shape: (12296, 35), y shape: (12296,), positive rate: 0.250


In [11]:
# Cell 8 — Train/test split and train A' logistic regression

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

clf_aprime = LogisticRegression(max_iter=1000, random_state=42)
clf_aprime.fit(X_train, y_train)

y_prob = clf_aprime.predict_proba(X_test)[:, 1]
roc = roc_auc_score(y_test, y_prob)
pr = average_precision_score(y_test, y_prob)

print(f"\nA' (35D) Results:")
print(f"  ROC-AUC: {roc:.3f}")
print(f"  PR-AUC:  {pr:.3f}")
print(f"\nFor reference:")
print(f"  A2 (31D, spatial only):     ROC-AUC 0.931, PR-AUC 0.842")
print(f"  A3 (32D, scalar PPE delta): ROC-AUC 0.950, PR-AUC 0.868")

Train: (9836, 35), Test: (2460, 35)

A' (35D) Results:
  ROC-AUC: 0.937
  PR-AUC:  0.855

For reference:
  A2 (31D, spatial only):     ROC-AUC 0.931, PR-AUC 0.842
  A3 (32D, scalar PPE delta): ROC-AUC 0.950, PR-AUC 0.868


In [12]:
# Cell 9 — Save A' model and log results

with open(STAGE5_BASE + "stage5_Aprime_logistic.pkl", "wb") as f:
    pickle.dump(clf_aprime, f)
print("Saved clf_aprime")

# Summary
print("""
Stage 5.1 — Spatiotemporal Embedding (A') Summary
--------------------------------------------------
Feature vector: [Vf (15D), Vp (15D), N (1D), V_delta (4D)] = 35D
V_delta: PCA of per-plant delta matrix (3160 bins × 52 weeks), 4 components
Variance explained by V_delta PCA: 38.7% (20.1, 10.9, 5.2, 2.5%)

Results:
  ROC-AUC: 0.937  (+0.006 vs A2, -0.013 vs A3)
  PR-AUC:  0.855  (+0.013 vs A2, -0.013 vs A3)

Finding: Scalar PPE delta (A3) outperforms spatiotemporal matrix embedding (A').
Likely reason: A3's scalar encodes plant-pollinator phenological alignment directly
(min of flowering × activity curves per pair). A' captures plant-side spatiotemporal
structure only — pollinator temporal signal is lost in the 4D compression.
""")

Saved clf_aprime

Stage 5.1 — Spatiotemporal Embedding (A') Summary
--------------------------------------------------
Feature vector: [Vf (15D), Vp (15D), N (1D), V_delta (4D)] = 35D
V_delta: PCA of per-plant delta matrix (3160 bins × 52 weeks), 4 components
Variance explained by V_delta PCA: 38.7% (20.1, 10.9, 5.2, 2.5%)

Results:
  ROC-AUC: 0.937  (+0.006 vs A2, -0.013 vs A3)
  PR-AUC:  0.855  (+0.013 vs A2, -0.013 vs A3)

Finding: Scalar PPE delta (A3) outperforms spatiotemporal matrix embedding (A').
Likely reason: A3's scalar encodes plant-pollinator phenological alignment directly
(min of flowering × activity curves per pair). A' captures plant-side spatiotemporal
structure only — pollinator temporal signal is lost in the 4D compression.

